In [1]:
import prettytable
import pandas as pd
import sqlite3

prettytable.DEFAULT = 'DEFAULT'

In [2]:
conn = sqlite3.connect('FinalDB.db')

In [6]:
%load_ext sql
%sql sqlite:///FinalDB.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [7]:
# Load CSV files into pandas DataFrames
census_df = pd.read_csv('ChicagoCensusData.csv')
crime_df = pd.read_csv('ChicagoCrimeData.csv')
schools_df = pd.read_csv('ChicagoPublicSchools.csv')

# Write the DataFrames into the SQLite database as tables
census_df.to_sql('ChicagoCensusData', conn, if_exists='replace', index=False)
crime_df.to_sql('ChicagoCrimeData', conn, if_exists='replace', index=False)
schools_df.to_sql('ChicagoPublicSchools', conn, if_exists='replace', index=False)

print('Loaded tables:', ', '.join(['ChicagoCensusData', 'ChicagoCrimeData', 'ChicagoPublicSchools']))

Loaded tables: ChicagoCensusData, ChicagoCrimeData, ChicagoPublicSchools


In [8]:
%%sql
SELECT COUNT(*) AS total_crimes
FROM ChicagoCrimeData;

 * sqlite:///FinalDB.db
Done.


total_crimes
533


In [9]:
%%sql
SELECT COMMUNITY_AREA_NUMBER,
       COMMUNITY_AREA_NAME,
       PER_CAPITA_INCOME
FROM ChicagoCensusData
WHERE PER_CAPITA_INCOME < 11000;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PER_CAPITA_INCOME
26.0,West Garfield Park,10934
30.0,South Lawndale,10402
37.0,Fuller Park,10432
54.0,Riverdale,8201


In [10]:
%%sql
SELECT CASE_NUMBER
FROM ChicagoCrimeData
WHERE DESCRIPTION LIKE '%MINOR%';

 * sqlite:///FinalDB.db
Done.


CASE_NUMBER
HL266884
HK238408


In [11]:
%%sql
SELECT *
FROM ChicagoCrimeData
WHERE PRIMARY_TYPE = 'KIDNAPPING' AND DESCRIPTION LIKE '%CHILD%';

 * sqlite:///FinalDB.db
Done.


ID,CASE_NUMBER,DATE,BLOCK,IUCR,PRIMARY_TYPE,DESCRIPTION,LOCATION_DESCRIPTION,ARREST,DOMESTIC,BEAT,DISTRICT,WARD,COMMUNITY_AREA_NUMBER,FBICODE,X_COORDINATE,Y_COORDINATE,YEAR,LATITUDE,LONGITUDE,LOCATION
5276766,HN144152,2007-01-26,050XX W VAN BUREN ST,1792,KIDNAPPING,CHILD ABDUCTION/STRANGER,STREET,0,0,1533,15,29.0,25.0,20,1143050.0,1897546.0,2007,41.87490841,-87.75024931,"(41.874908413, -87.750249307)"


In [12]:
%%sql
SELECT DISTINCT PRIMARY_TYPE
FROM ChicagoCrimeData
WHERE LOCATION_DESCRIPTION LIKE '%SCHOOL%';

 * sqlite:///FinalDB.db
Done.


PRIMARY_TYPE
BATTERY
CRIMINAL DAMAGE
NARCOTICS
ASSAULT
CRIMINAL TRESPASS
PUBLIC PEACE VIOLATION


In [14]:
%%sql
SELECT "Elementary, Middle, or High School", AVG(SAFETY_SCORE) AS avg_safety_score
FROM ChicagoPublicSchools
GROUP BY "Elementary, Middle, or High School";

 * sqlite:///FinalDB.db
Done.


"Elementary, Middle, or High School",avg_safety_score
ES,49.52038369304557
HS,49.62352941176471
MS,48.0


In [16]:
%%sql
SELECT COMMUNITY_AREA_NAME, PERCENT_HOUSEHOLDS_BELOW_POVERTY
FROM ChicagoCensusData
ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC
LIMIT 5;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME,PERCENT_HOUSEHOLDS_BELOW_POVERTY
Riverdale,56.5
Fuller Park,51.2
Englewood,46.6
North Lawndale,43.1
East Garfield Park,42.4


In [17]:
%%sql
SELECT COMMUNITY_AREA_NUMBER
FROM ChicagoCrimeData
GROUP BY COMMUNITY_AREA_NUMBER
ORDER BY COUNT(*) DESC
LIMIT 1;

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NUMBER
25.0


In [21]:
%%sql
SELECT COMMUNITY_AREA_NAME
FROM ChicagoCensusData
WHERE HARDSHIP_INDEX IN (SELECT MAX(HARDSHIP_INDEX) FROM ChicagoCensusData);

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME
Riverdale


In [19]:
%%sql
SELECT COMMUNITY_AREA_NAME
FROM ChicagoCensusData
WHERE COMMUNITY_AREA_NUMBER = (
    SELECT COMMUNITY_AREA_NUMBER
    FROM ChicagoCrimeData
    GROUP BY COMMUNITY_AREA_NUMBER
    ORDER BY COUNT(*) DESC
    LIMIT 1
);

 * sqlite:///FinalDB.db
Done.


COMMUNITY_AREA_NAME
Austin
